# POI Cleaning - Silver Layer

Cleans and structures raw POI data from Bronze layer.

**Purpose**: Transform raw OSM POI data into cleaned structured format.

**Input**: `{catalog}.{bronze_schema}.raw_pois`

**Outputs**:
- `{catalog}.{silver_schema}.clean_pois` - All cleaned POIs
- `{catalog}.{silver_schema}.pois_partners` - Partner stores (co-location opportunities: Walmart, Speedway, 7-Eleven, Shaw's)
- `{catalog}.{silver_schema}.pois_competitors` - Pizza competitors (exclusion zones)

In [0]:
import yaml
from pyspark.sql import functions as F
from pyspark.sql.types import *
from datetime import datetime

# Notebook parameters
dbutils.widgets.text("catalog", "")
dbutils.widgets.text("bronze_schema", "")
dbutils.widgets.text("silver_schema", "")
dbutils.widgets.text("config_path", "")
dbutils.widgets.text("state_filter", "")

# Extract parameters
catalog = dbutils.widgets.get("catalog")
bronze_schema = dbutils.widgets.get("bronze_schema")
silver_schema = dbutils.widgets.get("silver_schema")
config_path = dbutils.widgets.get("config_path")
state_filter = dbutils.widgets.get("state_filter")

assert catalog and bronze_schema and silver_schema and config_path, "Missing required parameters"

# Load configuration
with open(config_path, 'r') as f:
    config = yaml.safe_load(f)

poi_cleaning_config = config['poi_cleaning']

# Table names
input_table = f"{catalog}.{bronze_schema}.raw_pois"
output_table = f"{catalog}.{silver_schema}.clean_pois"

print(f"Input: {input_table}")
print(f"Output: {output_table}")
if state_filter:
    print(f"State filter: {state_filter}")

In [0]:
# Read raw POI data
pois_raw = spark.read.table(input_table)

# Validate table exists and has data
poi_count = pois_raw.count()

# Diagnostic: Show table info
table_info = spark.createDataFrame([
    ("Input table", input_table),
    ("Row count", str(poi_count))
], ["info", "value"])
display(table_info)

if poi_count == 0:
    raise RuntimeError(f"No POIs found in input table: {input_table}")

%md
## Extract and Clean Columns


In [ ]:
# Extract category and subcategory from tags using UDF - more reliable for Python dict handling
category_priority = poi_cleaning_config.get('category_priority', ['shop', 'amenity', 'leisure', 'tourism', 'office', 'public_transport', 'railway'])

def get_category_and_subcategory(tags):
    """Extract category and subcategory from tags dict
    Returns (category, subcategory) tuple
    Example: {'amenity': 'pharmacy'} -> ('amenity', 'pharmacy')
    """
    if tags is None or not isinstance(tags, dict):
        return (None, None)
    
    # Check each priority tag in order
    for category_tag in category_priority:
        if category_tag in tags:
            tag_value = tags[category_tag]
            if tag_value and str(tag_value).strip():
                return (category_tag, str(tag_value).strip())
    
    return (None, None)

# Create UDF with struct return type
category_schema = StructType([
    StructField("category", StringType(), True),
    StructField("subcategory", StringType(), True)
])
get_category_udf = F.udf(get_category_and_subcategory, category_schema)

# Extract address components using configured address fields
address_fields = poi_cleaning_config.get('address_fields', ['addr:housenumber', 'addr:street', 'addr:city', 'addr:state', 'addr:postcode'])

def build_address(tags):
    """Build address string from tags using configured address fields"""
    if tags is None or not isinstance(tags, dict):
        return None
    
    parts = []
    for field in address_fields:
        if field in tags and tags[field]:
            parts.append(str(tags[field]))
    
    return ', '.join(parts) if parts else None

build_address_udf = F.udf(build_address, StringType())

# Extract city from tags
def extract_city(tags):
    """Extract city from tags"""
    if tags is None or not isinstance(tags, dict):
        return None
    return tags.get('addr:city')

extract_city_udf = F.udf(extract_city, StringType())

# Extract state from tags
def extract_state(tags):
    """Extract state from tags"""
    if tags is None or not isinstance(tags, dict):
        return None
    return tags.get('addr:state')

extract_state_udf = F.udf(extract_state, StringType())

# Clean POI data
poi_id_prefix = poi_cleaning_config.get('poi_id_prefix', 'poi_')
pois_with_category = pois_raw \
    .withColumn("poi_id", F.concat(F.lit(poi_id_prefix), F.col("osm_id"))) \
    .withColumn("name", F.col("tags")["name"]) \
    .withColumn("category_struct", get_category_udf(F.col("tags"))) \
    .withColumn("poi_category", F.col("category_struct.category")) \
    .withColumn("poi_subcategory", F.col("category_struct.subcategory")) \
    .withColumn("latitude", F.col("latitude").cast("double")) \
    .withColumn("longitude", F.col("longitude").cast("double")) \
    .withColumn("address", build_address_udf(F.col("tags"))) \
    .withColumn("city", extract_city_udf(F.col("tags"))) \
    .withColumn("state", extract_state_udf(F.col("tags"))) \
    .withColumn("ingestion_timestamp", F.lit(datetime.now()))

# Apply filters and select final columns
pois_cleaned = pois_with_category \
    .select(
        "poi_id",
        "name",
        "poi_category",
        "poi_subcategory",
        "latitude",
        "longitude",
        "address",
        "city",
        "state",
        "osm_id",
        "osm_type",
        "ingestion_timestamp"
    ) \
    .filter(
        F.col("latitude").isNotNull() &
        F.col("longitude").isNotNull() &
        F.col("poi_category").isNotNull()
    )

%md
## Validate Data Quality


In [0]:
# Validate coordinate bounds using configured bounds
coord_bounds = poi_cleaning_config.get('coordinate_bounds', {
    'latitude_min': -90,
    'latitude_max': 90,
    'longitude_min': -180,
    'longitude_max': 180
})

pois_cleaned = pois_cleaned.filter(
    (F.col("latitude") >= coord_bounds['latitude_min']) & 
    (F.col("latitude") <= coord_bounds['latitude_max']) &
    (F.col("longitude") >= coord_bounds['longitude_min']) & 
    (F.col("longitude") <= coord_bounds['longitude_max'])
)

display(pois_cleaned.limit(10))

%md
## Write to Silver Table


In [ ]:
# Write to Silver table
pois_cleaned.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .option("delta.autoOptimize.optimizeWrite", "true") \
    .option("delta.autoOptimize.autoCompact", "true") \
    .saveAsTable(output_table)

# Summary statistics
summary = spark.sql(f"""
    SELECT 
        COUNT(*) as total_pois,
        COUNT(DISTINCT poi_category) as poi_categories,
        COUNT(DISTINCT poi_subcategory) as poi_subcategories,
        COUNT(DISTINCT osm_type) as osm_types,
        COUNT(CASE WHEN name IS NOT NULL THEN 1 END) as pois_with_name,
        COUNT(CASE WHEN address IS NOT NULL THEN 1 END) as pois_with_address,
        COUNT(CASE WHEN city IS NOT NULL THEN 1 END) as pois_with_city,
        COUNT(CASE WHEN state IS NOT NULL THEN 1 END) as pois_with_state
    FROM {output_table}
""")

display(summary)

## Split POIs into Partners and Competitors

In [ ]:
# Define brand patterns for splitting
# Partners (co-location opportunities) - stores where Little Caesars kiosks could be placed
# Use EXACT matching to avoid false positives like "Charles River Speedway/Notch Brewery"
partner_brands = [
    'walmart',
    'walmart supercenter',
    'speedway',
    '7-eleven',
    "shaw's",
]

# Competitors (exclusion zones for Little Caesars)
competitor_brands = ['pizza hut', "domino's", "papa john's", 'papa johns']

# Read the cleaned POIs
pois_all = spark.table(output_table)

# Filter for partner stores using EXACT matching (case-insensitive)
# This avoids false positives like "Charles River Speedway/Notch Brewery"
partner_brands_lower = [b.lower() for b in partner_brands]
pois_partners = pois_all.filter(
    F.lower(F.trim(F.col("name"))).isin(partner_brands_lower) &
    F.col("address").isNotNull()
)

# Add partner_brand column for normalized brand names
# Maps variant names to master brands for filtering in frontend
pois_partners = pois_partners.withColumn(
    "partner_brand",
    F.when(
        F.lower(F.col("name")).contains("walmart"), F.lit("Walmart")
    ).when(
        F.lower(F.col("name")).isin(["7-eleven", "speedway"]), F.lit("7-Eleven/Speedway")
    ).when(
        F.lower(F.col("name")).contains("shaw"), F.lit("Shaw's")
    ).otherwise(F.lit("Other"))
)

# Filter for competitors (with valid address for accuracy)
competitor_pattern = '|'.join(competitor_brands)
pois_competitors = pois_all.filter(
    F.lower(F.col("name")).rlike(competitor_pattern) &
    F.col("address").isNotNull()
)

# Output table names
partners_table = f"{catalog}.{silver_schema}.pois_partners"
competitors_table = f"{catalog}.{silver_schema}.pois_competitors"

print(f"Partner brands: {partner_brands}")
print(f"Competitor brands: {competitor_brands}")

In [ ]:
# Write partners table
(pois_partners
 .write
 .format("delta")
 .mode("overwrite")
 .option("overwriteSchema", "true")
 .saveAsTable(partners_table))

# Write competitors table
(pois_competitors
 .write
 .format("delta")
 .mode("overwrite")
 .option("overwriteSchema", "true")
 .saveAsTable(competitors_table))

# Get counts from written tables (single scan each)
partner_count = spark.table(partners_table).count()
comp_count = spark.table(competitors_table).count()

print(f"Written {partner_count} partner stores to {partners_table}")
print(f"Written {comp_count} competitors to {competitors_table}")

# Display samples
print("\nSample Partner Stores:")
display(spark.table(partners_table).limit(10))

print("\nSample Competitors:")
display(spark.table(competitors_table).limit(5))

# Show partner store breakdown by brand
print("\nPartner stores by brand:")
display(spark.table(partners_table).groupBy("name").count().orderBy(F.desc("count")).limit(10))